In [10]:
import formulallm.formula as f

import io
import contextlib

from llama_index.core.tools import FunctionTool
from llama_index.core.agent import ReActAgent, FunctionCallingAgentWorker
from llama_index.llms.ollama import Ollama

In [2]:
llama3 = Ollama(model="llama3", base_url='http://localhost:11434', temperature=0.0, request_timeout=600)

In [3]:
def load(filePath: str) -> str:
    """Load the Formula DSL code from the file path"""
    code = f.load(filePath)
    return code

load_tool = FunctionTool.from_defaults(fn=load)

In [11]:
def solve():
    """Try to solve the partial model based on the domain constraints"""
    stdout_buffer = io.StringIO()
    stderr_buffer = io.StringIO()

    with contextlib.redirect_stdout(stdout_buffer), contextlib.redirect_stderr(stderr_buffer):
        try:
            f.solve("pm","1","Mapping.conforms")
        except Exception as e:
            pass

    message = stdout_buffer.getvalue()
    error_message = stderr_buffer.getvalue()

    return message + "\n" + error_message

solve_tool = FunctionTool.from_defaults(fn=solve)

In [12]:
def extract(task_id: str):
    """Extract the solve result to the task with task_id.
    If the partial model is solvable, will return the solution.
    Else if the partial model is unsolvable, will return the least unsatisfied core conditions.
    The task_id is a string representing an integer which is incremented by 1 each time we run the solve command."""
    
    stdout_buffer = io.StringIO()
    stderr_buffer = io.StringIO()

    with contextlib.redirect_stdout(stdout_buffer), contextlib.redirect_stderr(stderr_buffer):
        try:
            f.extract(task_id, "0", "0")
        except Exception as e:
            pass

    message = stdout_buffer.getvalue()
    error_message = stderr_buffer.getvalue()

    return message + "\n" + error_message

extract_result_tool = FunctionTool.from_defaults(fn=extract)

In [13]:
react_agent = ReActAgent.from_tools(tools=[load_tool, solve_tool, extract_result_tool], llm=llama3, verbose=True)

In [14]:
response = react_agent.chat("Can you help me load the file from file path './data/MappingExample.4ml'")

print(str(response))

> Running step 16e58676-59c7-4c0c-b51c-b6fa0a9988bd. Step input: Can you help me load the file from file path './data/MappingExample.4ml'
Thought: The current language of the user is: English. I need to use a tool to help me answer the question.
Action: load
Action Input: {'filePath': './data/MappingExample.4ml'}
(Compiled) MappingExample.4ml
0.06s.
Observation: domain Mapping
{
  Component ::= new (id: Integer, utilization: Real).
  Processor ::= new (id: Integer).
  Mapping   ::= new (c: Component, p: Processor).

  // The utilization must be > 10
  invalidUtilization :- c is Component, c.utilization <= 10.

  badMapping :- p is Processor, 
		s = sum(0.0, { c.utilization |
			       c is Component, Mapping(c, p) }), s > 100.

  conforms no badMapping, no invalidUtilization.
}

partial model pm of Mapping
{
  c1 is Component(0, x).
  c2 is Component(1, y).
  p1 is Processor(0).
  p2 is Processor(1).
  Mapping(c1, p1).
  Mapping(c2, p1).
}
> Running step e5d32be2-7fa5-4e4a-b07b-ffa1c3f

In [15]:
response = react_agent.chat('solve the loaded partial model using the solve tool.You should not pass any parameter to the function.')

print(response)

> Running step 4639341b-07c3-47c1-97fa-69eba1651956. Step input: solve the loaded partial model using the solve tool.You should not pass any parameter to the function.
Thought: I need to use a tool to help me solve the partial model.
Action: solve
Action Input: {}
Observation: Parsing text took: 0
Visiting text took: 0
Started solve task with Id 1.
0.04s.


> Running step 1ce8e501-f81a-4445-b1b2-16b877472c34. Step input: None
Thought: The solve tool has been used to try to solve the partial model based on the domain constraints.
Action: extract
Action Input: {'task_id': '1'}
Observation: Solution number 0
Processor(0)
Processor(1)
Component(0, 11)
Component(1, 11)
Mapping(Component(0, 11), Processor(0))
Mapping(Component(1, 11), Processor(0))


0.00s.


> Running step 3253e532-3bc9-4e71-ac7b-d27e27b0624f. Step input: None
Observation: Error: Could not parse output. Please follow the thought-action-input format. Try again.
> Running step a590050f-bce3-431e-92a8-945d51660081. Step input: